<a href="https://colab.research.google.com/github/Sebastianelli-Nicola/Traffic-Driven-EV-Queuing-Predictor/blob/main/reconstruction_missing_period.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ricostruzione e Imputazione Avanzata di Serie Storiche di Traffico

Il presente notebook illustra il processo completo di imputazione e ricostruzione dei dati mancanti (*Data Imputation*) all'interno di serie storiche relative al traffico veicolare.

Si applica un algoritmo customizzato basato sulla ricerca di periodi analoghi (simil-KNN) e sull'iniezione di varianza locale, ottimizzato tramite Grid Search bidirezionale. L'obiettivo è colmare gap temporali estesi garantendo la coerenza statistica, la preservazione dei pattern orari pendolari e l'adattamento ai cali fisiologici legati alla stagionalità (es. periodo estivo).

Il processo culmina con una validazione avanzata che analizza le metriche di errore e la forma delle distribuzioni sia su scala Macro (mensile) che Micro (settimane adiacenti).

## 1. Importazione Librerie e Pre-elaborazione Dati
In questa prima fase si importano le librerie necessarie (Pandas, NumPy, Seaborn, Matplotlib, ecc.) e si effettua il montaggio dell'ambiente Google Drive per l'accesso ai file. Successivamente, si caricano i dataset storici (traffico e anagrafica stazioni). Si eseguono le operazioni di pulizia strutturale, il parsing rigoroso dei formati data/ora e la creazione della variabile target `differenza` (traffico libero vs attuale), assicurandosi di imporre il limite fisico dello zero.

In [ ]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from google.colab import drive

# ==========================================
# 1. CARICAMENTO E PRE-ELABORAZIONE
# ==========================================
print("Montaggio Google Drive e caricamento dati...")
drive.mount('/content/drive')

path_traffico = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_originale/traffico.csv'
path_stazioni = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_originale/stazioni.csv'

df_tr = pd.read_csv(path_traffico)
df_st = pd.read_csv(path_stazioni)

df_tr.columns = df_tr.columns.str.strip()
df_st.columns = df_st.columns.str.strip()

df_tr['giorno'] = pd.to_datetime(df_tr['giorno'], errors='coerce')
df_tr['orario'] = df_tr['orario'].astype(str).str.strip()
df_tr['timestamp'] = pd.to_datetime(df_tr['giorno'].astype(str) + " " + df_tr['orario'], errors='coerce')

df_tr['velocita_attuale'] = pd.to_numeric(df_tr['velocita_attuale'], errors='coerce')
df_tr['velocita_senza_traffico'] = pd.to_numeric(df_tr['velocita_senza_traffico'], errors='coerce')

df_tr['differenza'] = df_tr['velocita_senza_traffico'] - df_tr['velocita_attuale']
df_tr['differenza'] = df_tr['differenza'].clip(lower=0)

df = df_tr.merge(df_st, left_on='id_stazione', right_on='id', how='left')
df = df.dropna(subset=['timestamp', 'differenza']).sort_values(['id_stazione', 'timestamp']).reset_index(drop=True)

df['month'] = df['timestamp'].dt.month
df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.weekday

## 2. Funzioni Core dell'Algoritmo (Ricostruzione)
Qui si definiscono le funzioni matematiche centrali dell'algoritmo:
1. **Ricerca Periodi Analoghi (`find_analogous_periods`)**: Seleziona i giorni storici più simili utilizzando una gerarchia di punteggi rigida (Ora > Giorno > Recenza Bidirezionale).
2. **Ricostruzione per Campionamento (`reconstruct_station_by_sampling`)**: Calcola il valore atteso aggregando i candidati scelti e iniettando una perturbazione gaussiana basata sulla sola varianza locale. Applica un limite calcolato sulla media dei picchi pre e post gap.
3. **Cross-Validation a 1 Settimana (`grid_search_reconstruction`)**: Cerca i parametri ottimali simulando buchi finti nei giorni strettamente adiacenti al gap reale.
4. **Sfumatura (`smooth_transitions`)**: Garantisce la continuità della curva ai bordi del gap.

In [ ]:
# ==========================================
# 2. FUNZIONI CORE CON GERARCHIA RIGIDA
# ==========================================

def find_analogous_periods(df_valid, target_timestamp, gap_start, gap_end, n_candidates=15):
    target_weekday = target_timestamp.weekday()
    target_hour = target_timestamp.hour

    hours = df_valid['hour'].values
    weekdays = df_valid['weekday'].values
    diffs = df_valid['differenza'].values

    # Calcolo distanza temporale dai bordi del buco
    target_ns = target_timestamp.value
    start_ns = gap_start.value
    end_ns = gap_end.value

    mid_point_ns = start_ns + (end_ns - start_ns) / 2
    timestamps_ns = df_valid['timestamp'].values.astype(np.int64)

    if target_ns <= mid_point_ns:
        days_diff = np.abs(timestamps_ns - start_ns) / (1e9 * 3600 * 24)
    else:
        days_diff = np.abs(timestamps_ns - end_ns) / (1e9 * 3600 * 24)

    scores = np.zeros(len(df_valid))

    # 1. REGOLA D'ORO: L'Ora esatta (+100)
    scores += np.where(hours == target_hour, 100, 0)

    # 2. REGOLA D'ARGENTO: Il Giorno esatto (+50)
    scores += np.where(weekdays == target_weekday, 50, 0)

    # 3. TIE-BREAKER: Recenza (Decade gradualmente da +20 a 0)
    # Essendo < 50, non scambierà MAI un Lunedì per una Domenica pur di avere recenza.
    scores += np.maximum(0, 20 - (days_diff / 3))

    # Bonus di tolleranza per ore limitrofe (es. le 07:50 per le 08:00)
    hour_diff = np.abs(hours - target_hour)
    hour_diff = np.minimum(hour_diff, 24 - hour_diff)
    scores += np.maximum(0, 10 - hour_diff)

    top_indices = np.argsort(-scores)[:n_candidates]
    best_diffs = diffs[top_indices]

    # Outlier filter locale
    mean_val = np.mean(best_diffs)
    std_val = np.std(best_diffs)
    if std_val > 0:
        mask = (best_diffs >= mean_val - 2*std_val) & (best_diffs <= mean_val + 2*std_val)
        filtered = best_diffs[mask]
        if len(filtered) >= 5:
            return filtered

    return best_diffs

def reconstruct_station_by_sampling(df_station, gap_start, gap_end, freq, n_candidates=15, n_sample=5):
    missing_timestamps = pd.date_range(start=gap_start + freq, end=gap_end - freq, freq=freq)
    if len(missing_timestamps) == 0:
        return pd.DataFrame()

    df_valid = df_station[
        (df_station['month'] != 8) &
        ((df_station['timestamp'] < gap_start) | (df_station['timestamp'] > gap_end))
    ].copy()

    if df_valid.empty:
        return pd.DataFrame({'timestamp': missing_timestamps, 'differenza': 0})

    # TETTO FISICO: Media dei Picchi Adiacenti (1 Settimana)
    pre_week = df_valid[(df_valid['timestamp'] >= gap_start - pd.Timedelta(days=7)) & (df_valid['timestamp'] < gap_start)]
    post_week = df_valid[(df_valid['timestamp'] > gap_end) & (df_valid['timestamp'] <= gap_end + pd.Timedelta(days=7))]

    max_pre = pre_week['differenza'].max() if not pre_week.empty else np.nan
    max_post = post_week['differenza'].max() if not post_week.empty else np.nan

    # Media matematica dei due picchi. Ignora automaticamente eventuali NaN.
    with np.errstate(invalid='ignore'):
        tetto_assoluto = np.nanmean([max_pre, max_post])

    if pd.isna(tetto_assoluto) or tetto_assoluto <= 0:
        tetto_assoluto = df_valid['differenza'].max() # Fallback

    reconstructed_values = []
    fallback_mean = df_valid['differenza'].mean()

    for ts in missing_timestamps:
        candidates = find_analogous_periods(df_valid, ts, gap_start, gap_end, n_candidates=n_candidates)

        if candidates is not None and len(candidates) > 0:
            n_samp = min(n_sample, len(candidates))
            base_value = np.mean(candidates[:n_samp])
            local_sigma = np.std(candidates[:n_samp])
        else:
            base_value = fallback_mean
            local_sigma = 0

        noise = np.random.normal(loc=0, scale=local_sigma) if local_sigma > 0 else 0

        value = base_value + noise
        # Sicurezza: Niente traffico negativo, e blocco fermo sulla media dei picchi
        value = np.clip(value, 0, tetto_assoluto)

        reconstructed_values.append(value)

    return pd.DataFrame({'timestamp': missing_timestamps, 'differenza': reconstructed_values})

def grid_search_reconstruction(df_station, gap_start, gap_end, freq, param_grid):
    test_gaps = []

    # CV a 7 giorni
    test_start_pre = gap_start - pd.Timedelta(days=7)
    test_end_pre = gap_start
    test_start_post = gap_end
    test_end_post = gap_end + pd.Timedelta(days=7)

    if len(df_station[(df_station['timestamp'] >= test_start_pre) & (df_station['timestamp'] < test_end_pre)]) > 100:
        test_gaps.append((test_start_pre, test_end_pre))

    if len(df_station[(df_station['timestamp'] > test_start_post) & (df_station['timestamp'] <= test_end_post)]) > 100:
        test_gaps.append((test_start_post, test_end_post))

    if not test_gaps:
        print("      [!] Dati di test adiacenti insufficienti, uso default.")
        return {'n_candidates': 15, 'n_sample': 5}

    best_mae = float('inf')
    best_params = None
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    for params in combinations:
        maes_for_this_param = []
        for t_start, t_end in test_gaps:
            mask_test = (df_station['timestamp'] > t_start) & (df_station['timestamp'] < t_end)
            df_test_real = df_station[mask_test]

            if df_test_real.empty:
                continue

            rec_test = reconstruct_station_by_sampling(
                df_station, t_start, t_end, freq,
                n_candidates=params['n_candidates'], n_sample=params['n_sample']
            )

            merged = pd.merge(df_test_real[['timestamp', 'differenza']], rec_test, on='timestamp', suffixes=('_true', '_pred'))
            if not merged.empty:
                mae = mean_absolute_error(merged['differenza_true'], merged['differenza_pred'])
                maes_for_this_param.append(mae)

        if maes_for_this_param:
            avg_mae = np.mean(maes_for_this_param)
            print(f"      [Test] Parametri {params} -> MAE: {avg_mae:.2f}")
            if avg_mae < best_mae:
                best_mae = avg_mae
                best_params = params

    return best_params if best_params else combinations[0]

def smooth_transitions(df_station, gap_start, gap_end, reconstructed, n_blend=24):
    if reconstructed.empty:
        return reconstructed

    pre_gap = df_station[(df_station['timestamp'] < gap_start) &
                         (df_station['timestamp'] >= gap_start - pd.Timedelta(hours=n_blend))].sort_values('timestamp')
    post_gap = df_station[(df_station['timestamp'] > gap_end) &
                          (df_station['timestamp'] <= gap_end + pd.Timedelta(hours=n_blend))].sort_values('timestamp')

    values = reconstructed['differenza'].values.copy()

    if len(pre_gap) > 0:
        n_blend_start = min(n_blend, len(values) // 4)
        pre_mean = pre_gap['differenza'].tail(5).mean()
        for i in range(n_blend_start):
            weight = i / n_blend_start
            values[i] = (1 - weight) * pre_mean + weight * values[i]

    if len(post_gap) > 0:
        n_blend_end = min(n_blend, len(values) // 4)
        post_mean = post_gap['differenza'].head(5).mean()
        for i in range(n_blend_end):
            idx = len(values) - n_blend_end + i
            weight = i / n_blend_end
            values[idx] = (1 - weight) * values[idx] + weight * post_mean

    reconstructed['differenza'] = values
    return reconstructed

## 3. Identificazione Gap ed Esecuzione
In questa sezione si calcola automaticamente l'intervallo temporale mancante globale. Si itera poi su ogni singola stazione, applicando prima la Grid Search per individuare i migliori iperparametri locali e avviando successivamente l'imputazione. I dati ricostruiti vengono collezionati in un array globale.

In [ ]:
# ==========================================
# 3. IDENTIFICAZIONE GAP E ESECUZIONE
# ==========================================

stazioni = sorted(df['id_stazione'].unique())

df_sample = df[df['id_stazione'] == stazioni[0]].copy()
df_sample['time_diff'] = df_sample['timestamp'].diff()
freq = df_sample['time_diff'].mode()[0]
gap_idx = df_sample['time_diff'].idxmax()
gap_start = df_sample.loc[gap_idx - 1, 'timestamp']
gap_end = df_sample.loc[gap_idx, 'timestamp']

print(f"\nGap Globale Individuato: {gap_start} -> {gap_end}")

PARAM_GRID = {
    'n_candidates': [5, 10, 15, 20],
    'n_sample': [3, 5, 7]
}

all_reconstructed = []
print("\nInizio Ricostruzione con Gerarchia Rigida & Tetto Picchi Adiacenti...")

for st in stazioni:
    print(f"\n➜ Elaborazione Stazione {st}:")
    df_st = df[df['id_stazione'] == st].copy()

    best_params = grid_search_reconstruction(df_st, gap_start, gap_end, freq, PARAM_GRID)
    print(f"  [!] Scelto per Stazione {st} -> {best_params}")

    reconstructed = reconstruct_station_by_sampling(
        df_st, gap_start, gap_end, freq,
        n_candidates=best_params['n_candidates'],
        n_sample=best_params['n_sample']
    )

    if reconstructed.empty:
        continue

    reconstructed = smooth_transitions(df_st, gap_start, gap_end, reconstructed)
    reconstructed['id_stazione'] = st
    all_reconstructed.append(reconstructed)

## 4. Salvataggio e Grafici Panoramici
Una volta terminata l'elaborazione, le porzioni imputate vengono fuse con il dataset storico depurato da eventuali record corrotti interni al gap. Il dataframe finale (`df_final`) viene salvato come CSV sul Drive.
Vengono poi generati due blocchi di visualizzazione (matrici di grafici) per l'ispezione visiva della continuità della serie: uno zoom a 30 giorni e una prospettiva ampia a 90 giorni.

In [ ]:
# ==========================================
# 4. SALVATAGGIO E GRAFICI
# ==========================================
df_reconstructed = pd.concat(all_reconstructed, ignore_index=True)
df_clean = df[~((df['timestamp'] >= gap_start) & (df['timestamp'] <= gap_end))].copy()
df_final = pd.concat([df_clean[['timestamp', 'id_stazione', 'differenza']], df_reconstructed], ignore_index=True)
df_final = df_final.sort_values(['id_stazione', 'timestamp']).reset_index(drop=True)

df_final['differenza'] = df_final['differenza'].clip(lower=0).round(1)

print(f"\nRicostruzione Completata! Punti imputati: {len(df_reconstructed)}")

nome_file_output = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_ricostruito/traffico_r.csv'
df_final.to_csv(nome_file_output, index=False)
print(f"Dati salvati in: '{nome_file_output}'\n")

# ---- PLOT A 30 GIORNI ----
sns.set_theme(style="whitegrid")
num_stazioni = len(stazioni)
cols = 3
rows = int(np.ceil(num_stazioni / cols))
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(24, 5 * rows), sharex=True)
axes = np.array(axes).flatten()

plot_start = gap_start - pd.Timedelta(days=30)
plot_end = gap_end + pd.Timedelta(days=30)

for i, st in enumerate(stazioni):
    ax = axes[i]
    df_st = df_final[(df_final['id_stazione'] == st) & (df_final['timestamp'] >= plot_start) & (df_final['timestamp'] <= plot_end)].copy()
    mask_rec = (df_st['timestamp'] >= gap_start) & (df_st['timestamp'] <= gap_end)

    ax.plot(df_st[~mask_rec]['timestamp'], df_st[~mask_rec]['differenza'], label='Reali', color='#1f77b4', linewidth=1, alpha=0.7)
    ax.plot(df_st[mask_rec]['timestamp'], df_st[mask_rec]['differenza'], label='Ricostruiti', color='#ff7f0e', linewidth=1.2, alpha=0.9)

    ax.axvline(gap_start, color='red', linestyle='--', alpha=0.4)
    ax.axvline(gap_end, color='red', linestyle='--', alpha=0.4)
    ax.set_title(f'Stazione {st}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Diff. Velocità (km/h)')
    if i == 0: ax.legend(loc='upper left')

for i in range(num_stazioni):
    if i >= num_stazioni - cols:
        axes[i].xaxis.set_major_locator(mdates.DayLocator(interval=7))
        axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
        plt.setp(axes[i].get_xticklabels(), rotation=45)

for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
plt.suptitle('Panoramica Completa Ricostruzione (Gerarchia Ripristinata & Media Picchi) - Zoom 30 Gg', fontsize=20, y=1.02)
plt.tight_layout()
plt.show()

# ---- PLOT A 90 GIORNI ----
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(24, 5 * rows), sharex=True)
axes = np.array(axes).flatten()

plot_start = gap_start - pd.Timedelta(days=90)
plot_end = gap_end + pd.Timedelta(days=90)

for i, st in enumerate(stazioni):
    ax = axes[i]
    df_st = df_final[(df_final['id_stazione'] == st) & (df_final['timestamp'] >= plot_start) & (df_final['timestamp'] <= plot_end)].copy()
    mask_rec = (df_st['timestamp'] >= gap_start) & (df_st['timestamp'] <= gap_end)

    ax.plot(df_st[~mask_rec]['timestamp'], df_st[~mask_rec]['differenza'], label='Reali', color='#1f77b4', linewidth=1, alpha=0.7)
    ax.plot(df_st[mask_rec]['timestamp'], df_st[mask_rec]['differenza'], label='Ricostruiti', color='#ff7f0e', linewidth=1.2, alpha=0.9)

    ax.axvline(gap_start, color='red', linestyle='--', alpha=0.4)
    ax.axvline(gap_end, color='red', linestyle='--', alpha=0.4)
    ax.set_title(f'Stazione {st}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Diff. Velocità (km/h)')
    if i == 0: ax.legend(loc='upper left')

for i in range(num_stazioni):
    if i >= num_stazioni - cols:
        axes[i].xaxis.set_major_locator(mdates.DayLocator(interval=7))
        axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
        plt.setp(axes[i].get_xticklabels(), rotation=45)

for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
plt.suptitle('Panoramica Completa Ricostruzione (Gerarchia Ripristinata & Media Picchi) - Vista a 90 Gg', fontsize=20, y=1.02)
plt.tight_layout()
plt.show()